In [1]:
import polars as pl
from matplotlib import pyplot as plt
import seaborn as sns
import os

from pyspark.sql.dataframe import DataFrame
from pyspark.sql import SparkSession, functions
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType

os.chdir('../src')
print(os.getcwd())

/home/efran/GitHub/Spark-MLOps-Performance-Analysis/src


In [2]:
fraud_data = "/home/efran/GitHub/Spark-MLOps-Performance-Analysis/src/data/creditcardfraud/creditcard.csv"

def setup(data_path: str) -> (SparkSession, DataFrame):
    """
    Starts up spark session and loads fraud data into dataframe
    :param data_path: absolute path to fraud data
    :return: spark session and fraud data loaded in spark
    """
    spark: SparkSession = SparkSession.builder.appName(
        "CreditFraudDetector"
    ).getOrCreate()
    df: DataFrame = spark.read.csv(data_path, header=True, inferSchema=True)
    return spark, df
spark, credit_df = setup(fraud_data)

bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/24 20:45:39 WARN Utils: Your hostname, E-Desktop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/08/24 20:45:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/24 20:45:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
def preprocess(df: DataFrame) -> DataFrame:
    """
    Applies preprocessing step for ML pipeline
    :param df: spark df with credit card data loaded
    :return: dense vector with scaled data in "features_scaled" column
    """

    feature_cols = [
        "V1",
        "V2",
        "V3",
        "V4",
        "V5",
        "V6",
        "V7",
        "V8",
        "V9",
        "V10",
        "V11",
        "V12",
        "V13",
        "V14",
        "V15",
        "V16",
        "V17",
        "V18",
        "V19",
        "V20",
        "V21",
        "V22",
        "V23",
        "V24",
        "V25",
        "V26",
        "V27",
        "V28",
        "Amount",
        "Class",
    ]

    # Move data to dense vector. This vectorization process transforms data into format that ML model is trained on.
    # We select the relevant features we want to train our model on. In this case, thats all the columns minus time.
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    assembled_df = assembler.transform(df)
    # assembled_df.show(10, truncate=False)

    # TODO: add additional preprocessing steps to improve performance
    # Standard scaler normalizes features to a common scale. Normalization puts values into similar scale which can
    # help ML training because data points are more like for like.
    standard_scaler = StandardScaler(inputCol="features", outputCol="features_scaled")
    scaled_df = standard_scaler.fit(assembled_df).transform(assembled_df)
    # scaled_df.select("features", "features_scaled").show(10, truncate=False)

    return scaled_df

scaled_df = preprocess(credit_df)

25/08/24 20:45:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [35]:
scaled_df.count()

284807

In [54]:
RND_SEED = 123
def get_value_counts(df, col="Class"):
    value_counts_df = df.groupBy("Class").count()
    return value_counts_df

In [56]:
train_data, test_data = scaled_df.randomSplit([0.8, 0.2], seed=RND_SEED)
result_count = get_value_counts(train_data)
print("TRAIN", result_count.show())
result_count = get_value_counts(test_data)
print("TRAIN", result_count.show())

+-----+------+
|Class| count|
+-----+------+
|    1|   402|
|    0|227824|
+-----+------+

TRAIN None
+-----+-----+
|Class|count|
+-----+-----+
|    1|   90|
|    0|56491|
+-----+-----+

TRAIN None


In [53]:
test_df = scaled_df.sampleBy("Class", fractions= {0:1/5, 1:1/5}, seed=seed)

value_counts_df = test_df.groupBy("Class").count()

# Show the results
print(value_counts_df.show())

+-----+-----+
|Class|count|
+-----+-----+
|    1|  110|
|    0|57033|
+-----+-----+

None


+-----+-----+
|Class|count|
+-----+-----+
+-----+-----+

None
